# Exploration des données brutes

In [ ]:
import pandas as pd

RAW = "../data/raw/"
noms = ["taux_directeur_bce", "taux_credit_habitat", "volume_credit_habitat",
        "taux_credit_conso", "volume_credit_conso", "ipc_ensemble"]
data = {n: pd.read_csv(RAW + n + ".csv") for n in noms}

## Qualité des séries

In [ ]:
for n, df in data.items():
    print(n, len(df), "lignes |",
          "doublons :", df["TIME_PERIOD"].duplicated().sum(),
          "| nulls :", df["OBS_VALUE"].isna().sum(),
          "| statuts :", df["OBS_STATUS"].unique())

In [ ]:
# mois manquants (le taux directeur n'a des valeurs qu'aux dates de décision)
for n, df in data.items():
    if n == "taux_directeur_bce":
        continue
    mois = pd.period_range(df["TIME_PERIOD"].iloc[0], df["TIME_PERIOD"].iloc[-1], freq="M").astype(str)
    print(n, sorted(set(mois) - set(df["TIME_PERIOD"])))

## Taux directeur

In [ ]:
data["taux_directeur_bce"][["TIME_PERIOD", "OBS_VALUE"]]

## Inflation sur 12 mois

In [ ]:
ipc = data["ipc_ensemble"].set_index("TIME_PERIOD")["OBS_VALUE"]
inflation = (ipc / ipc.shift(12) - 1) * 100
inflation.dropna().round(2).loc[["2015-01", "2022-12", "2025-12", "2026-09"]]

## Taux réel (taux du crédit moins inflation)

In [ ]:
def serie(nom):
    return data[nom].set_index("TIME_PERIOD")["OBS_VALUE"]

df = pd.DataFrame({
    "taux_habitat": serie("taux_credit_habitat"),
    "taux_conso": serie("taux_credit_conso"),
    "volume_habitat": serie("volume_credit_habitat"),
    "inflation": inflation,
}).dropna()
df["reel_habitat"] = df["taux_habitat"] - df["inflation"]
df["reel_conso"] = df["taux_conso"] - df["inflation"]

negatifs = df[df["reel_habitat"] < 0]
print(len(negatifs), "mois négatifs, de", negatifs.index.min(), "à", negatifs.index.max())
print("minimum :", df["reel_habitat"].idxmin(), df["reel_habitat"].min().round(2))

In [ ]:
df.groupby(df.index.str[:4])[["taux_habitat", "inflation", "reel_habitat", "volume_habitat"]].mean().round(2)

## Saisonnalité du volume de crédit à l'habitat

In [ ]:
df.groupby(df.index.str[5:])["volume_habitat"].mean().round(0)